In [4]:
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
from sklearn.model_selection import train_test_split

print("="*60)
print(" RETRAINING XGBOOST WITH REAL TRAFFIC")
print("="*60)

# Load your Wireshark captures
wireshark_files = ['first_capture.csv', 'second_capture.csv']  # Add more files if you have them

real_traffic = []

for file in wireshark_files:
    try:
        df = pd.read_csv(file)
        print(f"Loaded {file}: {len(df)} packets")
        real_traffic.append(df)
    except:
        print(f"Could not load {file}")

# Combine all real traffic
if real_traffic:
    real_df = pd.concat(real_traffic, ignore_index=True)
    print(f"\nTotal real traffic samples: {len(real_df)}")
else:
    print("No Wireshark files found. Creating sample data...")
    # Create sample real traffic based on your results
    real_df = pd.DataFrame([
        {'proto': 'arp', 'service': 'arp', 'sbytes': 42, 'dbytes': 0, 'rate': 0.1, 'dur': 0.001},
        {'proto': 'arp', 'service': 'arp', 'sbytes': 42, 'dbytes': 0, 'rate': 0.1, 'dur': 0.001},
        {'proto': 'udp', 'service': 'dhcp', 'sbytes': 350, 'dbytes': 0, 'rate': 0.5, 'dur': 0.01},
        {'proto': 'udp', 'service': 'dns', 'sbytes': 92, 'dbytes': 257, 'rate': 10, 'dur': 0.01},
        {'proto': 'udp', 'service': 'dns', 'sbytes': 92, 'dbytes': 212, 'rate': 10, 'dur': 0.01},
    ])
    print(f"Created {len(real_df)} sample normal traffic records")

 RETRAINING XGBOOST WITH REAL TRAFFIC
Loaded first_capture.csv: 56 packets
Loaded second_capture.csv: 62 packets

Total real traffic samples: 118


In [5]:
def engineer_features_for_training(df):
    """Convert raw connection data to 19 features"""
    
    dangerous_protos = ['3pc', 'a/n', 'aes-sp3-d', 'any', 'argus']
    
    features_list = []
    for idx, row in df.iterrows():
        proto = str(row.get('proto', 'tcp')).lower()
        service = str(row.get('service', 'unknown')).lower()
        sbytes = float(row.get('sbytes', 500))
        dbytes = float(row.get('dbytes', 1000))
        rate = float(row.get('rate', 10))
        dur = float(row.get('dur', 1))
        
        features = {
            'is_sm_ips_ports': 0,
            'sbytes': sbytes,
            'dbytes': dbytes,
            'rate': rate,
            'dur': dur,
            'sload': 0,
            'dload': 0,
            'sinpkt': 0,
            'dinpkt': 0,
            'sjit': 0,
            'djit': 0,
            'tcprtt': 0,
            'synack': 0,
            'ackdat': 0,
            'bytes_ratio': sbytes / (dbytes + 1),
            'packets_ratio': 1.0,
            'load_ratio': 1.0,
            'jitter_product': 0,
            'dangerous_proto': 1 if proto in dangerous_protos else 0
        }
        features_list.append(features)
    
    return pd.DataFrame(features_list)

# Engineer features for real traffic
real_features = engineer_features_for_training(real_df)
real_features['label'] = 0  # All real traffic is NORMAL
print(f"Real traffic features shape: {real_features.shape}")

Real traffic features shape: (118, 20)


In [6]:
# Load original UNSW training data
X_train_orig = pd.read_csv('data/processed/X_train_final.csv')
y_train_orig = pd.read_csv('data/processed/y_train.csv').values.ravel()

print(f"Original training data: {X_train_orig.shape}")
print(f"Original attack samples: {sum(y_train_orig==1)}")
print(f"Original normal samples: {sum(y_train_orig==0)}")

Original training data: (125973, 18)
Original attack samples: 58630
Original normal samples: 67343


In [7]:
# Match columns between datasets
# Get the feature columns from original data
feature_cols = X_train_orig.columns.tolist()

# Ensure real_features has same columns
for col in feature_cols:
    if col not in real_features.columns:
        real_features[col] = 0

# Combine
X_combined = pd.concat([X_train_orig, real_features[feature_cols]], ignore_index=True)
y_combined = np.concatenate([y_train_orig, real_features['label'].values])

print(f"\n Combined dataset: {X_combined.shape}")
print(f"   Total normal samples: {sum(y_combined==0)}")
print(f"   Total attack samples: {sum(y_combined==1)}")


 Combined dataset: (126091, 18)
   Total normal samples: 67461
   Total attack samples: 58630


In [8]:
print("\n" + "="*60)
print(" RETRAINING XGBOOST WITH MIXED DATASET")
print("="*60)

# Split into train/validation
X_train, X_val, y_train, y_val = train_test_split(
    X_combined, y_combined, test_size=0.2, random_state=42, stratify=y_combined
)

# Train new model
xgb_model_new = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    scale_pos_weight=1.5,  # Adjust for imbalance
    random_state=42,
    eval_metric='logloss'
)

xgb_model_new.fit(X_train, y_train)
print(" Model retrained!")

# Save new model
joblib.dump(xgb_model_new, 'models/UNSW/xgboost_mixed.pkl')
print(" Model saved to: models/UNSW/xgboost_mixed.pkl")

# Quick validation
y_pred_val = xgb_model_new.predict(X_val)
from sklearn.metrics import recall_score, precision_score
print(f"\n Validation Performance:")
print(f"   Recall: {recall_score(y_val, y_pred_val):.4f}")
print(f"   Precision: {precision_score(y_val, y_pred_val):.4f}")


 RETRAINING XGBOOST WITH MIXED DATASET
 Model retrained!
 Model saved to: models/UNSW/xgboost_mixed.pkl

 Validation Performance:
   Recall: 0.9987
   Precision: 0.9987


In [9]:
print("\n" + "="*60)
print(" TESTING ON YOUR WIRESHARK CAPTURES")
print("="*60)

# Test on the real traffic
real_pred = xgb_model_new.predict(real_features[feature_cols])
real_proba = xgb_model_new.predict_proba(real_features[feature_cols])[:, 1]

print(f"\nPredictions on your real traffic:")
print(f"   ATTACK predictions: {sum(real_pred==1)}")
print(f"   NORMAL predictions: {sum(real_pred==0)}")

print(f"\nConfidence scores:")
for i, prob in enumerate(real_proba[:10]):
    pred = "ATTACK" if prob >= 0.80 else "NORMAL"
    print(f"   Sample {i+1}: {pred} ({prob:.2%})")


 TESTING ON YOUR WIRESHARK CAPTURES

Predictions on your real traffic:
   ATTACK predictions: 0
   NORMAL predictions: 118

Confidence scores:
   Sample 1: NORMAL (0.01%)
   Sample 2: NORMAL (0.01%)
   Sample 3: NORMAL (0.01%)
   Sample 4: NORMAL (0.01%)
   Sample 5: NORMAL (0.01%)
   Sample 6: NORMAL (0.01%)
   Sample 7: NORMAL (0.01%)
   Sample 8: NORMAL (0.01%)
   Sample 9: NORMAL (0.01%)
   Sample 10: NORMAL (0.01%)


In [12]:
# Load UNSW test data
X_test = pd.read_csv('data/processed/X_test_optimized.csv')
y_test = pd.read_csv('data/processed/y_test_final.csv').values.ravel()

print(f"Original test data features: {X_test.shape[1]}")

# ============================================
# REMOVE 'jitter_product' to match new model (18 features)
# ============================================

# Your new model expects 18 features (without jitter_product)
# Check which feature to remove
print(f"\nFeatures in X_test: {list(X_test.columns)}")

# Remove jitter_product if it exists
if 'jitter_product' in X_test.columns:
    X_test = X_test.drop(columns=['jitter_product'])
    print(f"\n Removed 'jitter_product'")
    print(f"New test data features: {X_test.shape[1]}")
else:
    print("\n 'jitter_product' not found in test data")

# Verify the model's expected features
print(f"\nModel expects: {xgb_model_new.get_booster().feature_names}")

# Verify we have the correct features
model_features = xgb_model_new.get_booster().feature_names
X_test = X_test[model_features]  # Reorder to match model exactly

print(f"\n Test data now has {X_test.shape[1]} features (matching model)")

# Predict
test_pred = xgb_model_new.predict(X_test)
test_proba = xgb_model_new.predict_proba(X_test)[:, 1]

from sklearn.metrics import recall_score, precision_score

print(f"\n UNSW Test Set Performance:")
print(f"   Recall: {recall_score(y_test, test_pred):.4f}")
print(f"   Precision: {precision_score(y_test, test_pred):.4f}")
print(f"   Attacks caught: {sum((test_pred==1) & (y_test==1))}")
print(f"   False alarms: {sum((test_pred==1) & (y_test==0))}")

Original test data features: 19

Features in X_test: ['is_sm_ips_ports', 'sbytes', 'dbytes', 'rate', 'dur', 'sload', 'dload', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'tcprtt', 'synack', 'ackdat', 'bytes_ratio', 'packets_ratio', 'load_ratio', 'jitter_product', 'dangerous_proto']

 Removed 'jitter_product'
New test data features: 18

Model expects: ['same_srv_rate', 'src_bytes', 'dst_host_srv_count', 'flag', 'dst_host_same_srv_rate', 'logged_in', 'srv_serror_rate', 'serror_rate', 'dst_host_srv_serror_rate', 'dst_host_serror_rate', 'count', 'service', 'dst_bytes', 'diff_srv_rate', 'dst_host_diff_srv_rate', 'bytes_ratio', 'total_bytes', 'error_ratio']


KeyError: "['same_srv_rate', 'src_bytes', 'dst_host_srv_count', 'flag', 'dst_host_same_srv_rate', 'logged_in', 'srv_serror_rate', 'serror_rate', 'dst_host_srv_serror_rate', 'dst_host_serror_rate', 'count', 'service', 'dst_bytes', 'diff_srv_rate', 'dst_host_diff_srv_rate', 'total_bytes', 'error_ratio'] not in index"